<a href="https://colab.research.google.com/github/sergkurilenko/DeepML/blob/HW1/DeepML_HW1_Kurilenko.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Домашнее задание 1 (HW) Куриленко Сергей

In [2]:
# импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [3]:
# Фиксируем сиды для воспроизводимости
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

1. Загрузка данных и диагностика

In [4]:
# Пути к данным
TRAIN_PATH = 'fmnist_train.csv'
TEST_PATH  = 'fmnist_test.csv'

# Загрузка
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)

# Проверка и заполнение NaN
print("NaNs in train before fill:", train_df.isna().sum().sum())
train_df.fillna(0, inplace=True)
print("NaNs in train after fill:", train_df.isna().sum().sum())
# Тестовых NaN нет, но на всякий случай
print("NaNs in test before fill: ", test_df.isna().sum().sum())
test_df.fillna(0, inplace=True)
print("NaNs in test after fill: ", test_df.isna().sum().sum())

# Распределение меток
print("Label distribution:", train_df['label'].value_counts().sort_index().to_dict())

Train shape: (17040, 786)
Test shape:  (10000, 785)
NaNs in train before fill: 424
NaNs in train after fill: 0
NaNs in test before fill:  0
NaNs in test after fill:  0
Label distribution: {0: 1770, 1: 1700, 2: 1677, 3: 1725, 4: 1639, 5: 1695, 6: 1704, 7: 1761, 8: 1675, 9: 1694}


2. Подготовка данных

In [5]:
# Удаляем лишние колонки
cols_to_drop = ['label', 'Id']
X = train_df.drop(cols_to_drop, axis=1).values.astype('float32') / 255.0
y = to_categorical(train_df['label'].values, num_classes=10)

# Тестовые данные (drop Id)
X_test = test_df.drop(['Id'], axis=1).values.astype('float32') / 255.0

# Преобразуем в формат изображений
X = X.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# Разделение на обучающую/валидационную выборку с стратификацией
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print("Train/Val shapes:", X_train.shape, X_val.shape)

Train/Val shapes: (13632, 28, 28, 1) (3408, 28, 28, 1)


3. Реализуем CNN с аугментацией

In [6]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(),
    Dropout(0.25),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(),
    Dropout(0.25),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')
])
model.compile('adam', 'categorical_crossentropy', metrics=['accuracy'])

datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
datagen.fit(X_train)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


4. Обучение

In [7]:
# Обучение
EPOCHS = 100
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=256),
    epochs=EPOCHS,
    validation_data=(X_val, y_val)
)

Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


54/54 ━━━━━━━━━━━━━━━━━━━━ 19s 174ms/step - accuracy: 0.2975 - loss: 2.1815 - val_accuracy: 0.1080 - val_loss: 2.2758
Epoch 2/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 63ms/step - accuracy: 0.6154 - loss: 1.0768 - val_accuracy: 0.1335 - val_loss: 2.3943
Epoch 3/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 63ms/step - accuracy: 0.6731 - loss: 0.8825 - val_accuracy: 0.1112 - val_loss: 2.6593
Epoch 4/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 64ms/step - accuracy: 0.7089 - loss: 0.7799 - val_accuracy: 0.1426 - val_loss: 2.7008
Epoch 5/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 64ms/step - accuracy: 0.7136 - loss: 0.7555 - val_accuracy: 0.1546 - val_loss: 2.8319
Epoch 6/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 64ms/step - accuracy: 0.7403 - loss: 0.6953 - val_accuracy: 0.2482 - val_loss: 1.8365
Epoch 7/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 64ms/step - accuracy: 0.7507 - loss: 0.6662 - val_accuracy: 0.3991 - val_loss: 1.5793
Epoch 8/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 3s 64ms/step - accuracy: 0.7606 - loss: 0.6384 - val_accuracy: 0.6253 - val

5. Оценка на валидации

In [8]:
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print("Final Validation Accuracy:", val_acc)

Final Validation Accuracy: 0.8999413251876831


6. Сохраняем сабмишн по образцу

In [9]:
SUB_PATH = 'sample_submission.csv'
preds = model.predict(X_test)
submission = pd.read_csv(SUB_PATH)
submission['label'] = np.argmax(preds, axis=1)
submission.to_csv('submission.csv', index=False)
print("Submission saved.")




313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Submission saved.
